In [1]:
import numpy as np 
import pickle
import pandas as pd

In [26]:
data = pickle.load(open('data_face_features_test.pickel', mode='rb'))
data1 = pickle.load(open('data_face_features_train.pickel', mode='rb'))

In [27]:
X=np.array(data1['data'])
Y= np.array(data1['label'])

In [32]:
X= X.reshape(-1,128)
X.shape

(7124, 128)

In [33]:
from sklearn.model_selection import train_test_split


In [34]:
x_train,x_test,y_train,y_test= train_test_split(X,Y,train_size=0.8,random_state=42)  

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score




In [36]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# x_train = scaler.fit_transform(x_train)
# x_test = scaler.transform(x_test)


In [37]:
model_logistic= LogisticRegression(max_iter=5000,multi_class='multinomial',
    solver='lbfgs')
model_logistic.fit(x_train,y_train)

/home/usman/Documents/Robotics_AI/Some_projects/reinforcement_learning/rl_env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'multinomial'


In [10]:
def get_report(model,x_train,y_train,x_test,y_test):
    y_pred_train = model.predict(x_train)
    y_pred_test= model.predict(x_test)
    acc_train = accuracy_score(y_train,y_pred_train)
    acc_test= accuracy_score(y_test,y_pred_test)


    f1_train = f1_score(y_train,y_pred_train,average='macro')
    f1_test= f1_score(y_test,y_pred_test,average='macro')
    print('Accuracy Train =', acc_train)
    print('Accuracy Test =' ,acc_test)
    print('F1_score_train Train =' , f1_train)
    print('F1_score_train Test =' , f1_test)

In [38]:
get_report(model_logistic,x_train,y_train,x_test,y_test)

Accuracy Train = 0.37918933146165995
Accuracy Test = 0.37543859649122807
F1_score_train Train = 0.28431138889088475
F1_score_train Test = 0.28090803318997704


In [39]:
model_SVC= SVC(    kernel='rbf',
    C=15,
    gamma=0.01,
    probability=True)
model_SVC.fit(x_train,y_train)

,C,15
,kernel,'rbf'
,degree,3
,gamma,0.01
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [40]:
get_report(model_SVC,x_train,y_train,x_test,y_test)


Accuracy Train = 0.37146867871556416
Accuracy Test = 0.3726315789473684
F1_score_train Train = 0.26748663419326035
F1_score_train Test = 0.27257765767795206


In [41]:
model_rf= RandomForestClassifier()
model_rf.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [42]:
get_report(model_rf,x_train,y_train,x_test,y_test)


Accuracy Train = 0.9994735918582207
Accuracy Test = 0.3607017543859649
F1_score_train Train = 0.9994910746073912
F1_score_train Test = 0.2868598692375963


In [43]:
model_voting= VotingClassifier(estimators=[
    ('logistic', LogisticRegression()),
    ('svm',SVC(probability=True)),
    ('rf', RandomForestClassifier())
],voting='soft',weights=[2,3,1])
model_voting.fit(x_train,y_train)

,estimators,"[('logistic', ...), ('svm', ...), ...]"
,voting,'soft'
,weights,"[2, 3, ...]"
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [44]:
get_report(model_voting,x_train,y_train,x_test,y_test)


Accuracy Train = 0.6813476048429549
Accuracy Test = 0.39649122807017545
F1_score_train Train = 0.6061519256304425
F1_score_train Test = 0.30625302239423485


In [45]:
from sklearn.model_selection import GridSearchCV

In [46]:
model_grid = GridSearchCV(model_voting,
                          param_grid={
                              'svm__C':[3,5,7,10],
                              'svm__gamma':[0.1,0.3,0.5],
                              'rf__n_estimators':[5,10,20],
                              'rf__max_depth':[3,5,7],
                              'voting':['soft','hard']
                          },scoring='accuracy',cv=3,n_jobs=1,verbose=2)

In [47]:
model_grid.fit(x_train,y_train)

Fitting 3 folds for each of 216 candidates, totalling 648 fits


[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  25.0s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  26.2s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=soft; total time=  27.0s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  24.8s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  25.5s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.1, voting=hard; total time=  27.4s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  26.6s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  26.1s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gamma=0.3, voting=soft; total time=  26.7s
[CV] END rf__max_depth=3, rf__n_estimators=5, svm__C=3, svm__gam

,estimator,"VotingClassif...hts=[2, 3, 1])"
,param_grid,"{'rf__max_depth': [3, 5, ...], 'rf__n_estimators': [5, 10, ...], 'svm__C': [3, 5, ...], 'svm__gamma': [0.1, 0.3, ...], ...}"
,scoring,'accuracy'
,n_jobs,1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [52]:
get_report(model_grid,x_train,y_train,x_test,y_test)


Accuracy Train = 0.4879803474293736
Accuracy Test = 0.39859649122807017
F1_score_train Train = 0.3846485292874827
F1_score_train Test = 0.3037487991262763


In [53]:
model_best_estimator= model_grid.best_estimator_

In [54]:
model_grid.best_score_

np.float64(0.38076817900464693)

In [55]:
pickle.dump(model_best_estimator,open('./models/machinelearning_emotions.pkl',mode='wb'))